Permeation formula

$$ln J = ln K + ln D − ln h + ln C_v$$

1. Diffusion in the stratum corneum (Fick's 2nd law)
$$\frac{\partial C}{\partial t} = D\frac{\partial^{2} C}{\partial x^{2}}$$
2. SC surface (vehicle side)
$$C(0,t) = K \times S_w \times a(t) \times f_{un}, a(t) = \frac{C_v(t)}{C_{sat}}$$
- a(t) = activity: how full the vehicle is, from 0 to 1.
- f_un = unionised share of the drug, from pKa and pH.
3. SC bottom (receptor = sink)
4. Receptor (IVPT measures)

# 0 - Setup

In [8]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

SEED = 678
RNG = np.random.default_rng(SEED) # fixed seed -> same "random" data every run
OUT = Path("mechanistic_out"); OUT.mkdir(exist_ok=True)

In [3]:
# function helper -- general

def show(name):
    # save the figure AND display it
    plt.tight_layout()
    plt.savefig(OUT / f"{name}.png", dpi=120)
    plt.show()

pd.set_option("display.width", 200); pd.set_option("display.precision", 3)
PCOL = {"caffeine": "#2a78d6", "benzoic_acid": "#1baf7a", "testosterone": "#eb6834"}

# 1 - Formula descriptors

In [7]:
# function helper -- analysis

def frac_unionised(pKa, pH):
    # weak acid: Henderson-Hasselbalch. Neutral molecule -> 1
    return 1.0 if np.isnan(pKa) else 1 / (1 + 10 ** (pH - pKa))

### A - The actives

In [6]:
PROBES = pd.DataFrame({
    "MW":   [194.19, 122.12, 288.42], # g/mol
    "logP": [-0.07,  1.87,   3.32], # octanol/water
    "pKa":  [np.nan, 4.20,   np.nan], # NaN = stays neutral at pH 4-5 (caffeine's pKa ~0.6)
    "S_w":  [21600., 3400.,  23.4], # water solubility at 25 C, ug/cm3 (= mg/L)
}, index=["caffeine", "benzoic_acid", "testosterone"]) # alternative for the liphopilic?
PROBE_NAMES = list(PROBES.index)

print(PROBES)

                  MW  logP  pKa      S_w
caffeine      194.19 -0.07  NaN  21600.0
benzoic_acid  122.12  1.87  4.2   3400.0
testosterone  288.42  3.32  NaN     23.4


### B - The vehicle/formula

# 2 - Skin descriptors